# Data Science Case Study 1: AutoML with Genetic Algorithms

## 🎯 Objetivo

Construir un sistema **AutoML básico** usando Algoritmos Genéticos para:
1. Seleccionar el mejor modelo
2. Optimizar hiperparámetros
3. Seleccionar features relevantes
4. Todo de forma automática!

## 📚 Lo que Aprenderás

- Pipeline optimization con GAs
- Multi-level optimization (modelo + hiperparámetros + features)
- Codificación de pipelines complejos
- Evaluación robusta con cross-validation
- Comparación con Grid Search y Random Search

## 🔧 Caso Real

**Problema:** Clasificación de calidad de vino
- 11 features químicas
- Múltiples modelos posibles (RF, SVM, XGBoost, etc.)
- Muchos hiperparámetros
- Features pueden ser redundantes

**Objetivo:** Encontrar el mejor pipeline automáticamente

---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.datasets import load_wine, load_breast_cancer, load_digits
import sys
import time

sys.path.append('../GA_Tutorial_1_Basics')
sys.path.append('../GA_Tutorial_2_Intermediate')
from ga_utils_basics import *
from ga_utils_intermediate import *

np.random.seed(42)
%matplotlib inline
plt.rcParams['figure.figsize'] = (14, 6)

print("✓ Imports successful!")

## 1. Cargar y Explorar Datos

In [ ]:
# Cargar dataset de vinos
data = load_wine()
X, y = data.data, data.target
feature_names = data.feature_names

print("="*70)
print("DATASET: Wine Quality Classification")
print("="*70)
print(f"Samples: {X.shape[0]}")
print(f"Features: {X.shape[1]}")
print(f"Classes: {len(np.unique(y))}")
print(f"\nFeature names: {feature_names[:5]}...")
print(f"Class distribution: {np.bincount(y)}")

# Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

print(f"\nTrain samples: {len(X_train)}")
print(f"Test samples: {len(X_test)}")

# Visualizar features
df = pd.DataFrame(X_train, columns=feature_names)
df['target'] = y_train

print("\nPrimeras filas:")
print(df.head())

## 2. Definir Espacio de Búsqueda de AutoML

### Pipeline Completo:
```
[Feature Selection] → [Model Selection] → [Hyperparameter Tuning]
```

### Codificación del Cromosoma:
```python
chromosome = [
    # Feature selection (13 bits, uno por feature)
    1, 0, 1, 1, 0, 1, 0, 1, 1, 0, 1, 0, 1,
    
    # Model selection (3 bits para 5 modelos)
    0, 1, 0,  # → SVM
    
    # Hyperparameters (resto)
    ...
]
```

In [ ]:
# Definir modelos y sus espacios de hiperparámetros
MODEL_SPACE = {
    'RandomForest': {
        'model': RandomForestClassifier,
        'params': {
            'n_estimators': (10, 200),
            'max_depth': (2, 20),
            'min_samples_split': (2, 20)
        }
    },
    'SVM': {
        'model': SVC,
        'params': {
            'C': (0.1, 100.0),
            'gamma': (0.001, 1.0)
        }
    },
    'KNN': {
        'model': KNeighborsClassifier,
        'params': {
            'n_neighbors': (3, 30),
            'weights': ['uniform', 'distance']
        }
    },
    'DecisionTree': {
        'model': DecisionTreeClassifier,
        'params': {
            'max_depth': (2, 20),
            'min_samples_split': (2, 20)
        }
    },
    'GradientBoosting': {
        'model': GradientBoostingClassifier,
        'params': {
            'n_estimators': (10, 200),
            'learning_rate': (0.01, 0.3),
            'max_depth': (2, 10)
        }
    }
}

print("Espacio de búsqueda AutoML:")
print(f"\nModelos disponibles: {len(MODEL_SPACE)}")
for model_name, config in MODEL_SPACE.items():
    print(f"  - {model_name}: {len(config['params'])} hiperparámetros")

print(f"\nFeatures disponibles: {X_train.shape[1]}")
print(f"\nEspacio de búsqueda total: ENORME! (por eso usamos GA)")

## 3. Implementar AutoML con GA

In [ ]:
def decode_automl_chromosome(chromosome, n_features):
    """
    Decodifica cromosoma a pipeline completo.
    
    Estructura:
    - [0:n_features]: feature selection (binario)
    - [n_features:n_features+3]: model selection (3 bits)
    - [resto]: hyperparameters (depende del modelo)
    """
    # 1. Feature selection
    feature_mask = chromosome[:n_features]
    selected_features = np.where(feature_mask > 0.5)[0]  # Real encoding
    
    if len(selected_features) == 0:
        selected_features = np.array([0])  # Al menos una feature
    
    # 2. Model selection (usar 3 genes para 5 modelos)
    model_genes = chromosome[n_features:n_features+3]
    model_idx = int(np.sum(model_genes) * len(MODEL_SPACE) / 3) % len(MODEL_SPACE)
    model_name = list(MODEL_SPACE.keys())[model_idx]
    model_config = MODEL_SPACE[model_name]
    
    # 3. Hyperparameters
    hyperparam_genes = chromosome[n_features+3:]
    hyperparams = {}
    
    param_names = list(model_config['params'].keys())
    genes_per_param = len(hyperparam_genes) // max(len(param_names), 1)
    
    for i, param_name in enumerate(param_names):
        start_idx = i * genes_per_param
        end_idx = start_idx + genes_per_param
        gene_value = np.mean(hyperparam_genes[start_idx:end_idx]) if end_idx <= len(hyperparam_genes) else 0.5
        
        param_range = model_config['params'][param_name]
        
        if isinstance(param_range, tuple):
            # Numeric parameter
            min_val, max_val = param_range
            if isinstance(min_val, int) and isinstance(max_val, int):
                hyperparams[param_name] = int(min_val + gene_value * (max_val - min_val))
            else:
                hyperparams[param_name] = min_val + gene_value * (max_val - min_val)
        else:
            # Categorical parameter
            idx = int(gene_value * len(param_range)) % len(param_range)
            hyperparams[param_name] = param_range[idx]
    
    return selected_features, model_name, hyperparams


def evaluate_automl_pipeline(chromosome, X_train, y_train, n_features, cv=3):
    """
    Evalúa pipeline completo con cross-validation.
    """
    try:
        # Decodificar
        selected_features, model_name, hyperparams = decode_automl_chromosome(chromosome, n_features)
        
        # Preparar datos
        X_selected = X_train[:, selected_features]
        
        # Crear modelo
        model_class = MODEL_SPACE[model_name]['model']
        model = model_class(**hyperparams, random_state=42)
        
        # Evaluar con CV
        scores = cross_val_score(model, X_selected, y_train, cv=cv, scoring='accuracy')
        
        # Penalizar por usar muchas features (regularización)
        feature_penalty = len(selected_features) / n_features * 0.05
        
        fitness = np.mean(scores) - feature_penalty
        
        return max(fitness, 0.0)
        
    except Exception as e:
        # Si el modelo falla, retornar fitness muy bajo
        return 0.0

print("✓ Funciones de AutoML definidas")

## 4. Ejecutar AutoML GA

In [ ]:
def automl_genetic_algorithm(X_train, y_train, pop_size=20, max_generations=30, cv=3):
    """
    AutoML usando Algoritmos Genéticos.
    """
    n_features = X_train.shape[1]
    chromosome_length = n_features + 3 + 12  # features + model + hyperparams
    
    # Inicializar población (encoding real [0,1])
    population = initialize_population_real(pop_size, chromosome_length, (0, 1))
    
    history = {
        'best_fitness': [],
        'avg_fitness': [],
        'best_pipeline': [],
        'diversity': []
    }
    
    print("="*70)
    print("AUTOML GENETIC ALGORITHM")
    print("="*70)
    print(f"Population size: {pop_size}")
    print(f"Max generations: {max_generations}")
    print(f"CV folds: {cv}")
    print(f"Search space: {len(MODEL_SPACE)} models × {n_features} features × hyperparams")
    print("\nEvolution progress:")
    print("-"*70)
    
    start_time = time.time()
    
    for generation in range(max_generations):
        # Evaluar fitness
        fitness_values = np.array([
            evaluate_automl_pipeline(ind, X_train, y_train, n_features, cv)
            for ind in population
        ])
        
        # Track best
        best_idx = np.argmax(fitness_values)
        best_chromosome = population[best_idx]
        
        selected_features, model_name, hyperparams = decode_automl_chromosome(best_chromosome, n_features)
        
        history['best_fitness'].append(np.max(fitness_values))
        history['avg_fitness'].append(np.mean(fitness_values))
        history['best_pipeline'].append({
            'features': selected_features,
            'model': model_name,
            'hyperparams': hyperparams
        })
        history['diversity'].append(calculate_diversity(population))
        
        if generation % 5 == 0 or generation == max_generations - 1:
            print(f"Gen {generation:2d} | Best: {np.max(fitness_values):.4f} | "
                  f"Avg: {np.mean(fitness_values):.4f} | "
                  f"Model: {model_name[:15]:15s} | Features: {len(selected_features):2d}/{n_features}")
        
        # Evolution
        parents = rank_based_selection(population, fitness_values, pop_size)
        
        offspring = []
        for i in range(0, pop_size, 2):
            p1, p2 = parents[i], parents[min(i+1, pop_size-1)]
            c1, c2 = simulated_binary_crossover(p1, p2, eta=20, bounds=(0, 1))
            c1 = polynomial_mutation(c1, 0.1, (0, 1), eta=20)
            c2 = polynomial_mutation(c2, 0.1, (0, 1), eta=20)
            offspring.extend([c1, c2])
        
        # Elitism
        population = np.array(offspring[:pop_size])
        population[0] = best_chromosome  # Keep best
    
    elapsed_time = time.time() - start_time
    
    print("-"*70)
    print(f"✓ AutoML completed in {elapsed_time:.1f} seconds")
    
    return best_chromosome, history

print("✓ AutoML GA definido")

In [ ]:
# Ejecutar AutoML
best_chromosome, history = automl_genetic_algorithm(
    X_train, y_train,
    pop_size=20,
    max_generations=30,
    cv=3
)

## 5. Analizar Mejor Pipeline Encontrado

In [ ]:
# Decodificar mejor pipeline
best_features, best_model_name, best_hyperparams = decode_automl_chromosome(
    best_chromosome, X_train.shape[1]
)

print("="*70)
print("MEJOR PIPELINE ENCONTRADO")
print("="*70)
print(f"\nModelo: {best_model_name}")
print(f"\nHiperparámetros:")
for param, value in best_hyperparams.items():
    print(f"  {param}: {value}")
print(f"\nFeatures seleccionadas: {len(best_features)}/{X_train.shape[1]}")
print(f"Índices: {best_features}")
print(f"Nombres: {[feature_names[i] for i in best_features[:5]]}...")

# Entrenar modelo final
X_train_selected = X_train[:, best_features]
X_test_selected = X_test[:, best_features]

model_class = MODEL_SPACE[best_model_name]['model']
final_model = model_class(**best_hyperparams, random_state=42)
final_model.fit(X_train_selected, y_train)

train_score = final_model.score(X_train_selected, y_train)
test_score = final_model.score(X_test_selected, y_test)

print(f"\n{'='*70}")
print("PERFORMANCE")
print("="*70)
print(f"Train accuracy: {train_score:.4f}")
print(f"Test accuracy:  {test_score:.4f}")
print(f"Generalization gap: {abs(train_score - test_score):.4f}")

## 6. Comparar con Baselines

In [ ]:
print("="*70)
print("COMPARACIÓN CON BASELINES")
print("="*70)

# Baseline 1: Random Forest con parámetros default y todas las features
baseline_rf = RandomForestClassifier(random_state=42)
baseline_rf.fit(X_train, y_train)
baseline_rf_score = baseline_rf.score(X_test, y_test)

print(f"\nBaseline 1 (RF default, todas features):  {baseline_rf_score:.4f}")

# Baseline 2: Mejor modelo individual (sin GA)
baseline_scores = []
for model_name, config in MODEL_SPACE.items():
    model = config['model'](random_state=42)
    model.fit(X_train, y_train)
    score = model.score(X_test, y_test)
    baseline_scores.append((model_name, score))

best_baseline = max(baseline_scores, key=lambda x: x[1])
print(f"Baseline 2 (Mejor modelo default):        {best_baseline[1]:.4f} ({best_baseline[0]})")

print(f"\n{'='*70}")
print(f"AutoML GA (optimizado):                   {test_score:.4f} ⭐")
print(f"{'='*70}")

improvement_vs_rf = ((test_score - baseline_rf_score) / baseline_rf_score) * 100
improvement_vs_best = ((test_score - best_baseline[1]) / best_baseline[1]) * 100

print(f"\nMejora vs RF default: {improvement_vs_rf:+.2f}%")
print(f"Mejora vs mejor baseline: {improvement_vs_best:+.2f}%")
print(f"Reducción de features: {(1 - len(best_features)/X_train.shape[1])*100:.1f}%")

## 7. Visualizar Evolución

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# 1. Fitness evolution
ax = axes[0, 0]
ax.plot(history['best_fitness'], 'g-', linewidth=2, label='Best Fitness')
ax.plot(history['avg_fitness'], 'b--', linewidth=1.5, label='Avg Fitness')
ax.axhline(y=test_score, color='r', linestyle=':', label=f'Final Test Score')
ax.set_xlabel('Generation')
ax.set_ylabel('Fitness (CV Accuracy)')
ax.set_title('AutoML Fitness Evolution')
ax.legend()
ax.grid(True, alpha=0.3)

# 2. Feature selection evolution
ax = axes[0, 1]
n_features_per_gen = [len(p['features']) for p in history['best_pipeline']]
ax.plot(n_features_per_gen, 'purple', linewidth=2)
ax.axhline(y=X_train.shape[1], color='gray', linestyle='--', label='Total features')
ax.set_xlabel('Generation')
ax.set_ylabel('Number of Selected Features')
ax.set_title('Feature Selection Over Time')
ax.legend()
ax.grid(True, alpha=0.3)

# 3. Model selection evolution
ax = axes[1, 0]
model_evolution = [p['model'] for p in history['best_pipeline']]
model_counts = pd.Series(model_evolution).value_counts()
model_counts.plot(kind='barh', ax=ax, color='steelblue')
ax.set_xlabel('Frequency as Best Model')
ax.set_title('Model Selection Frequency')
ax.grid(True, axis='x', alpha=0.3)

# 4. Diversity
ax = axes[1, 1]
ax.plot(history['diversity'], 'orange', linewidth=2)
ax.set_xlabel('Generation')
ax.set_ylabel('Population Diversity')
ax.set_title('Population Diversity Over Time')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n📊 Observaciones:")
print(f"  • Fitness mejoró de {history['best_fitness'][0]:.4f} a {history['best_fitness'][-1]:.4f}")
print(f"  • Features reducidas a {len(best_features)}/{X_train.shape[1]} ({len(best_features)/X_train.shape[1]*100:.1f}%)")
print(f"  • Modelo final: {best_model_name}")
print(f"  • Test accuracy: {test_score:.4f}")

## 8. Feature Importance Analysis

In [ ]:
# Si el modelo tiene feature_importances
if hasattr(final_model, 'feature_importances_'):
    importances = final_model.feature_importances_
    feature_importance_df = pd.DataFrame({
        'Feature': [feature_names[i] for i in best_features],
        'Importance': importances
    }).sort_values('Importance', ascending=False)
    
    plt.figure(figsize=(10, 6))
    plt.barh(feature_importance_df['Feature'], feature_importance_df['Importance'], color='steelblue')
    plt.xlabel('Importance')
    plt.title(f'Feature Importance - {best_model_name}')
    plt.gca().invert_yaxis()
    plt.grid(True, axis='x', alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    print("\nTop 5 features más importantes:")
    print(feature_importance_df.head())
else:
    print(f"\nModelo {best_model_name} no proporciona feature importance")
    print(f"Features seleccionadas: {[feature_names[i] for i in best_features]}")

## 🎯 Conclusiones del Caso de Estudio

### ✅ Lo que Logramos

1. **Optimización automática completa:**
   - Selección de modelo
   - Optimización de hiperparámetros  
   - Feature selection
   - Todo en un solo proceso!

2. **Ventajas del AutoML con GA:**
   - Explora espacio enorme eficientemente
   - Encuentra combinaciones no obvias
   - Balancea múltiples objetivos (accuracy vs complejidad)
   - Interpretable (puedes ver qué encontró)

3. **Resultados típicos:**
   - Mejor que modelos default
   - Menos features = más rápido + más interpretable
   - Comparable o mejor que Grid Search en menos tiempo

### 📚 Lecciones Aprendidas

1. **Codificación es clave:** Diseño del cromosoma determina qué puede encontrar
2. **Cross-validation esencial:** Evita overfitting
3. **Penalizaciones ayudan:** Regularizar complejidad (número de features)
4. **Múltiples runs:** GAs son estocásticos, correr varias veces

### 🚀 Próximos Pasos

1. Probar en tu propio dataset
2. Agregar más modelos (XGBoost, LightGBM)
3. Agregar preprocesamiento al pipeline
4. Implementar multi-objetivo (accuracy vs tiempo)
5. Guardar y deployar el mejor modelo

### 💡 Cuándo Usar AutoML con GA

✅ **Bueno para:**
- Datasets medianos (1K-100K samples)
- Cuando tienes tiempo de computación
- Problemas donde necesitas interpretar el pipeline
- Exploración inicial de modelos

❌ **Alternativas mejores:**
- Datasets muy grandes → AutoML tools (H2O, AutoKeras)
- Necesitas resultado en minutos → Random Search
- Deep learning → NAS específicos

---

**¡Felicidades!** Has implementado un sistema AutoML básico con GAs. 

Continúa con los siguientes casos de estudio para aprender más aplicaciones de GAs en Data Science! 🎉